In [31]:
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [32]:
INPUT_FILE = "../data/processed/final_dataset.csv"
MODEL_FILE = "../models/elastic_net_pipeline.pkl"

In [33]:
print("Loading final dataset...")

data = pd.read_csv(INPUT_FILE)

print("Dataset shape:", data.shape)

Loading final dataset...
Dataset shape: (5363, 27)


In [34]:
data.head()

,budget,log_budget,runtime,release_year,release_month,original_language,main_country,genre_action,genre_adventure,genre_animation,...,genre_horror,genre_music,genre_mystery,genre_romance,genre_science_fiction,genre_thriller,genre_war,genre_western,revenue,log_revenue
0,30000000.0,17.216708,81.0,1995.0,10.0,en,United States of America,0,0,1,...,0,0,0,0,0,0,0,0,373554033.0,19.738573
1,65000000.0,17.989898,104.0,1995.0,12.0,en,United States of America,0,1,0,...,0,0,0,0,0,0,0,0,262797249.0,19.386893
2,16000000.0,16.588099,127.0,1995.0,12.0,en,United States of America,0,0,0,...,0,0,0,1,0,0,0,0,81452156.0,18.215526
3,60000000.0,17.909855,170.0,1995.0,12.0,en,United States of America,1,0,0,...,0,0,0,0,0,1,0,0,187436818.0,19.048952
4,35000000.0,17.370859,106.0,1995.0,12.0,en,United States of America,1,1,0,...,0,0,0,0,0,1,0,0,64350171.0,17.979850


In [35]:
print("Dataset columns:")
print(data.columns.tolist())

Dataset columns:
['budget', 'log_budget', 'runtime', 'release_year', 'release_month', 'original_language', 'main_country', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_history', 'genre_horror', 'genre_music', 'genre_mystery', 'genre_romance', 'genre_science_fiction', 'genre_thriller', 'genre_war', 'genre_western', 'revenue', 'log_revenue']


In [36]:
target = "log_revenue"

numeric_features = [
    "budget",
    "log_budget",
    "runtime",
    "release_year",
    "release_month"
]

categorical_features = [
    "original_language",
    "main_country"
]

genre_features = [
    column
    for column in data.columns
    if column.startswith("genre_")
]

numeric_features.extend(genre_features)

X = data[
    numeric_features + categorical_features
]

y = data[target]

In [37]:
train_data = data[data["release_year"] <= 2015]
test_data = data[data["release_year"] > 2015]

In [38]:
X_train = train_data[numeric_features + categorical_features]
y_train = train_data["log_revenue"]

X_test = test_data[numeric_features + categorical_features]
y_test = test_data["log_revenue"]

In [39]:
print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(target)

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

Features:
['budget', 'log_budget', 'runtime', 'release_year', 'release_month', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_history', 'genre_horror', 'genre_music', 'genre_mystery', 'genre_romance', 'genre_science_fiction', 'genre_thriller', 'genre_war', 'genre_western', 'original_language', 'main_country']

Target:
log_revenue

Feature shape: (5363, 25)
Target shape: (5363,)


In [40]:
print("Missing values in features:")
print(X.isnull().sum())

print("\nMissing values in target:")
print(y.isnull().sum())

Missing values in features:
budget                   0
log_budget               0
runtime                  0
release_year             0
release_month            0
genre_action             0
genre_adventure          0
genre_animation          0
genre_comedy             0
genre_crime              0
genre_documentary        0
genre_drama              0
genre_family             0
genre_fantasy            0
genre_history            0
genre_horror             0
genre_music              0
genre_mystery            0
genre_romance            0
genre_science_fiction    0
genre_thriller           0
genre_war                0
genre_western            0
original_language        0
main_country             0
dtype: int64

Missing values in target:
0


In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4290
Testing samples: 1073


In [42]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [43]:
pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            ElasticNet(
                max_iter=10000
            )
        )
    ]
)

print("Elastic Net pipeline created.")

Elastic Net pipeline created.


In [44]:
parameters = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0
    ],
    "model__l1_ratio": [
        0.1,
        0.3,
        0.5,
        0.7,
        0.9
    ]
}

In [45]:
grid_search = GridSearchCV(
    pipeline,
    parameters,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

print("GridSearchCV created.")

GridSearchCV created.


In [46]:
print("Training Elastic Net...")

grid_search.fit(
    X_train,
    y_train
)

print("Training completed.")

Training Elastic Net...
Training completed.


In [47]:
print("Best Parameters:")
print(grid_search.best_params_)

Best Parameters:
{'model__alpha': 0.001, 'model__l1_ratio': 0.9}


In [48]:
print("Best CV Score:")
print(-grid_search.best_score_)

Best CV Score:
1.745726110550739


In [49]:
model = grid_search.best_estimator_

print("Best Elastic Net model selected.")

Best Elastic Net model selected.


In [50]:
predicted_log_revenue = model.predict(X_test)

print("Predictions generated.")
print("Number of predictions:", len(predicted_log_revenue))

Predictions generated.
Number of predictions: 1073


In [51]:
actual_revenue = np.expm1(y_test)

predicted_revenue = np.expm1(
    predicted_log_revenue
)

In [52]:
mae = mean_absolute_error(
    actual_revenue,
    predicted_revenue
)

print("MAE:", mae)

MAE: 54538514.81106156


In [53]:
rmse = np.sqrt(
    mean_squared_error(
        actual_revenue,
        predicted_revenue
    )
)

print("RMSE:", rmse)

RMSE: 123215559.78365093


In [54]:
r2 = r2_score(
    actual_revenue,
    predicted_revenue
)

print("R2:", r2)

R2: 0.31516697578509734


In [55]:
rmsle = np.sqrt(
    mean_squared_error(
        y_test,
        predicted_log_revenue
    )
)

print("RMSLE:", rmsle)

RMSLE: 1.8418729317168732


In [56]:
print("==============================")
print("MODEL EVALUATION")
print("==============================")

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("RMSLE:", rmsle)

MODEL EVALUATION
MAE: 54538514.81106156
RMSE: 123215559.78365093
R2: 0.31516697578509734
RMSLE: 1.8418729317168732


In [57]:
results = pd.DataFrame({
    "Metric": [
        "MAE",
        "RMSE",
        "R2",
        "RMSLE"
    ],
    "Value": [
        mae,
        rmse,
        r2,
        rmsle
    ]
})

results

,Metric,Value
0,MAE,5.453851e+07
1,RMSE,1.232156e+08
2,R2,3.151670e-01
3,RMSLE,1.841873e+00


In [58]:
comparison = pd.DataFrame({
    "Actual Revenue": actual_revenue.values,
    "Predicted Revenue": predicted_revenue
})

comparison.head(10)

,Actual Revenue,Predicted Revenue
0,5.000000e+05,1.649481e+06
1,9.797098e+06,1.108924e+07
2,8.417330e+05,2.478720e+06
3,2.023414e+06,3.340646e+06
4,2.335557e+08,7.866396e+06
5,2.658900e+07,6.011372e+06
6,3.200525e+07,3.014818e+07
7,2.048342e+07,4.267734e+06
8,2.851767e+08,7.823695e+07
9,1.576687e+06,2.068364e+07


In [59]:
joblib.dump(
    model,
    MODEL_FILE
)

print("Model saved successfully:")
print(MODEL_FILE)

Model saved successfully:
../models/elastic_net_pipeline.pkl


In [60]:
import os

if os.path.exists(MODEL_FILE):
    print("Model file exists.")
    print(MODEL_FILE)
else:
    print("Model file was not found.")

Model file exists.
../models/elastic_net_pipeline.pkl


In [61]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet

DATA_FILE = "../data/processed/model2_dataset.csv"

data = pd.read_csv(DATA_FILE)

print("Dataset shape:", data.shape)

Dataset shape: (5381, 12)


In [62]:
numeric_features = [
    "budget",
    "log_budget",
    "runtime",
    "release_year",
    "release_month",
    "cast_count",
    "production_company_count",
    "director_previous_movies"
]

categorical_features = [
    "original_language",
    "main_country"
]

genre_features = [
    column
    for column in data.columns
    if column.startswith("genre_")
]

numeric_features.extend(genre_features)

features = numeric_features + categorical_features

In [63]:
train_data = data[
    data["release_year"] <= 2015
].copy()

test_data = data[
    data["release_year"] > 2015
].copy()

X_train = train_data[features]
y_train = train_data["log_revenue"]

X_test = test_data[features]
y_test = test_data["log_revenue"]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 5075
Testing samples: 306


In [64]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [65]:
model = ElasticNet(
    max_iter=10000
)

pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ]
)

In [66]:
param_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0
    ],
    "model__l1_ratio": [
        0.1,
        0.3,
        0.5,
        0.7,
        0.9
    ]
}

In [67]:
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...iter=10000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.001, 0.01, ...], 'model__l1_ratio': [0.1, 0.3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of 

In [68]:
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest CV score:")
print(grid_search.best_score_)

Best parameters:
{'model__alpha': 0.01, 'model__l1_ratio': 0.1}

Best CV score:
0.4966571343795486


In [69]:
MODEL_FILE = "../models/elastic_net_model2.pkl"

joblib.dump(
    grid_search.best_estimator_,
    MODEL_FILE
)

print("\nModel 2 saved to:")
print(MODEL_FILE)


Model 2 saved to:
../models/elastic_net_model2.pkl


MODEL 3

In [71]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet

DATA_FILE = "../data/processed/model3_dataset.csv"

data = pd.read_csv(DATA_FILE)

print("Dataset shape:", data.shape)

Dataset shape: (5381, 11)


In [72]:
numeric_features = [
    "budget",
    "log_budget",
    "runtime",
    "release_year",
    "release_month",
    "director_previous_movies",
    "director_previous_avg_log_revenue"
]

categorical_features = [
    "original_language",
    "main_country"
]

features = numeric_features + categorical_features

In [73]:
train_data = data[
    data["release_year"] <= 2015
].copy()

test_data = data[
    data["release_year"] > 2015
].copy()

X_train = train_data[features]
y_train = train_data["log_revenue"]

X_test = test_data[features]
y_test = test_data["log_revenue"]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 5075
Testing samples: 306


In [74]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [75]:
model = ElasticNet(
    max_iter=10000
)

pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            model
        )
    ]
)

In [76]:
param_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0
    ],
    "model__l1_ratio": [
        0.1,
        0.3,
        0.5,
        0.7,
        0.9
    ]
}

In [77]:
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...iter=10000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__alpha': [0.001, 0.01, ...], 'model__l1_ratio': [0.1, 0.3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of 

In [78]:
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest CV score:")
print(grid_search.best_score_)

Best parameters:
{'model__alpha': 0.1, 'model__l1_ratio': 0.1}

Best CV score:
0.4907899795478426


In [79]:
MODEL_FILE = "../models/elastic_net_model3.pkl"

joblib.dump(
    grid_search.best_estimator_,
    MODEL_FILE
)

print("\nModel 3 saved to:")
print(MODEL_FILE)


Model 3 saved to:
../models/elastic_net_model3.pkl


MODEL 4

In [80]:
import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


INPUT_FILE = "../data/processed/model4_dataset.csv"
MODEL_FILE = "../models/elastic_net_model4.pkl"


data = pd.read_csv(INPUT_FILE)

print("Dataset shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

Dataset shape: (5381, 27)

Columns:
['budget', 'log_budget', 'runtime', 'release_year', 'release_month', 'original_language', 'main_country', 'genre_action', 'genre_adventure', 'genre_animation', 'genre_comedy', 'genre_crime', 'genre_documentary', 'genre_drama', 'genre_family', 'genre_fantasy', 'genre_history', 'genre_horror', 'genre_music', 'genre_mystery', 'genre_romance', 'genre_science_fiction', 'genre_thriller', 'genre_war', 'genre_western', 'revenue', 'log_revenue']


In [81]:
genre_columns = [
    column
    for column in data.columns
    if column.startswith("genre_")
]

numeric_features = [
    "budget",
    "log_budget",
    "runtime",
    "release_year",
    "release_month"
]

numeric_features.extend(genre_columns)

categorical_features = [
    "original_language",
    "main_country"
]

target = "log_revenue"

feature_columns = (
    numeric_features
    + categorical_features
)

X = data[feature_columns]
y = data[target]

print("Number of features:", len(feature_columns))
print("Target:", target)

Number of features: 25
Target: log_revenue


In [82]:
train_data = data[
    data["release_year"] <= 2015
]

test_data = data[
    data["release_year"] > 2015
]

X_train = train_data[feature_columns]
y_train = train_data[target]

X_test = test_data[feature_columns]
y_test = test_data[target]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print(
    "\nTraining years:",
    train_data["release_year"].min(),
    "to",
    train_data["release_year"].max()
)

print(
    "Testing years:",
    test_data["release_year"].min(),
    "to",
    test_data["release_year"].max()
)

Training data: (5075, 25)
Testing data: (306, 25)

Training years: 1915.0 to 2015.0
Testing years: 2016.0 to 2017.0


In [83]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            ElasticNet(
                max_iter=10000
            )
        )
    ]
)

In [84]:
parameter_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1.0,
        10.0
    ],
    "model__l1_ratio": [
        0.1,
        0.3,
        0.5,
        0.7,
        0.9
    ]
}

grid_search = GridSearchCV(
    pipeline,
    parameter_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

grid_search.fit(
    X_train,
    y_train
)

print("Best parameters:")
print(grid_search.best_params_)

print(
    "\nBest CV R2:",
    grid_search.best_score_
)

Best parameters:
{'model__alpha': 0.1, 'model__l1_ratio': 0.1}

Best CV R2: 0.4930117143287083


In [85]:
model4 = grid_search.best_estimator_

predicted_log_revenue = model4.predict(X_test)

predicted_revenue = np.expm1(
    predicted_log_revenue
)

actual_revenue = np.expm1(
    y_test
)

mae = mean_absolute_error(
    actual_revenue,
    predicted_revenue
)

rmse = np.sqrt(
    mean_squared_error(
        actual_revenue,
        predicted_revenue
    )
)

r2 = r2_score(
    actual_revenue,
    predicted_revenue
)

rmsle = np.sqrt(
    mean_squared_error(
        y_test,
        predicted_log_revenue
    )
)

print("Model 4 Results")
print("-------------------------")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)
print("RMSLE:", rmsle)

Model 4 Results
-------------------------
MAE: 83460551.04862802
RMSE: 168851393.20838276
R2: 0.49580030455372504
RMSLE: 1.6289945558925885


In [86]:
joblib.dump(
    model4,
    MODEL_FILE
)

print("\nModel 4 saved to:")
print(MODEL_FILE)


Model 4 saved to:
../models/elastic_net_model4.pkl
